# Synthetic slant-delay estimation

This notebook creates a nonlinear dolphin-like whistle whose instantaneous frequency follows the normalized contour $c(x)=\tan(x)-\sin(x)+1$ over $x\in[-1,1]$. It sends the whistle through a two-path impulse response, adds reproducible Gaussian noise, extracts the whistle's time-frequency ridge with the native Python contour extractor, reconstructs an analytic template, and estimates the separation between the direct and delayed arrivals with FFT matched filtering.

The generated WAV and the manually supplied interval are passed directly to `extract_birdcall_contours`. The accepted in-memory contour is used by the matched-filter stage; no MATLAB runtime or MAT file is required.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.io import wavfile
from IPython.display import Audio, display

from birdcall_contour_bundle import (
    birdcall_contour_default_config,
    extract_birdcall_contours,
)


## Configuration

All timing quantities use one sampling rate. The echo search starts after the configured minimum separation and ends at the maximum delay.

In [ ]:
FS = 48_000
SIGNAL_DURATION_S = 1
SIGNAL_X_MIN = -1.0
SIGNAL_X_MAX = 1.0
WHISTLE_START_HZ = 4_000
WHISTLE_END_HZ = 10_000
WHISTLE_TAPER_ALPHA = 0.25
TRUE_DELAY_S = 0.02
DIRECT_GAIN = 0.5
DELAYED_GAIN = 0.2
NOISE_STD = 0.01
RANDOM_SEED = 7

MIN_PEAK_SEPARATION_S = 0.002
MAX_DELAY_S = 0.030
MIN_PROMINENCE_RATIO = 0.10

CONTOUR_OUTPUT_DIR = "synthetic_dolphin_chirp_birdcall_contours"
MAX_DELAY_ERROR_SAMPLES = 2

## Generate the source and received signals

The $\tan(x)-\sin(x)+1$ curve is normalized and mapped to a 4--10 kHz instantaneous-frequency contour. Integrating that contour gives the phase of a nonlinear chirp, and a Tukey envelope supplies the smooth onset and offset typical of a whistle. The sparse filter contains a direct path at sample zero and a delayed path at the rounded delay sample.

In [ ]:
source_sample_count = round(SIGNAL_DURATION_S * FS)
source_time_s = np.arange(source_sample_count) / FS
source_x = np.linspace(SIGNAL_X_MIN, SIGNAL_X_MAX, source_sample_count)
contour_shape = np.tan(source_x) - np.sin(source_x) + 1
contour_normalized = (contour_shape - contour_shape.min()) / np.ptp(contour_shape)
instantaneous_frequency_hz = (
    WHISTLE_START_HZ
    + (WHISTLE_END_HZ - WHISTLE_START_HZ) * contour_normalized
)
phase_rad = 2 * np.pi * np.concatenate((
    [0.0], np.cumsum(instantaneous_frequency_hz[:-1]) / FS
))
amplitude_envelope = signal.windows.tukey(source_sample_count, alpha=WHISTLE_TAPER_ALPHA)
source = amplitude_envelope * np.sin(phase_rad)

true_delay_samples = round(TRUE_DELAY_S * FS)
impulse_response = np.zeros(true_delay_samples + 1)
impulse_response[0] = DIRECT_GAIN
impulse_response[true_delay_samples] = DELAYED_GAIN

received_clean = signal.convolve(source, impulse_response, mode="full")
rng = np.random.default_rng(RANDOM_SEED)
received_noisy = received_clean + NOISE_STD * rng.standard_normal(received_clean.size)
received_time_s = np.arange(received_noisy.size) / FS

### Listen to the generated signal

Run the following cell and use the inline player to hear the synthetic source signal.

In [ ]:
display(Audio(source, rate=FS, normalize=True))

### Save the signal as a WAV file

A peak-normalized copy is converted to signed 16-bit PCM and saved in the notebook's working directory.

In [ ]:
wav_path = "synthetic_dolphin_chirp.wav"
source_for_audio = source / np.max(np.abs(source))
source_pcm16 = np.int16(source_for_audio * np.iinfo(np.int16).max)
wavfile.write(wav_path, FS, source_pcm16)
print(f"Saved {wav_path} at {FS} Hz ({source.size / FS:.3f} s).")

### Spectrogram of the generated signal

The dashed line is the requested nonlinear dolphin-like frequency contour; the spectrogram should track it from 4 to 10 kHz.

In [ ]:
spectrogram_frequency_hz, spectrogram_time_s, spectrogram_magnitude = signal.spectrogram(
    source,
    fs=FS,
    window="hann",
    nperseg=512,
    noverlap=384,
    nfft=1024,
    scaling="spectrum",
    mode="magnitude",
)
spectrogram_db = 20 * np.log10(
    np.maximum(spectrogram_magnitude, np.finfo(float).tiny)
)

fig, ax = plt.subplots(figsize=(11, 4.5), constrained_layout=True)
mesh = ax.pcolormesh(
    spectrogram_time_s * 1e3,
    spectrogram_frequency_hz / 1e3,
    spectrogram_db,
    shading="auto",
    cmap="magma",
)
ax.set(
    title="Nonlinear dolphin-like whistle contour",
    xlabel="Time (ms)",
    ylabel="Frequency (kHz)",
)
ax.plot(
    source_time_s * 1e3,
    instantaneous_frequency_hz / 1e3,
    color="cyan",
    linestyle="--",
    linewidth=1.2,
    label="Target contour",
)
ax.set_ylim((WHISTLE_START_HZ - 1_000) / 1e3, (WHISTLE_END_HZ + 1_000) / 1e3)
ax.legend(loc="upper left")
fig.colorbar(mesh, ax=ax, label="Magnitude (dB)")
plt.show()

In [ ]:
# Lightweight construction checks
assert source.size == round(SIGNAL_DURATION_S * FS)
assert np.isclose(source_x[0], -1.0) and np.isclose(source_x[-1], 1.0)
assert np.isclose(instantaneous_frequency_hz[0], WHISTLE_START_HZ)
assert np.isclose(instantaneous_frequency_hz[-1], WHISTLE_END_HZ)
assert np.all(np.diff(instantaneous_frequency_hz) > 0)
assert np.max(np.abs(source)) <= 1.0
assert true_delay_samples == round(TRUE_DELAY_S * FS)
assert impulse_response.size == true_delay_samples + 1
assert np.flatnonzero(impulse_response).tolist() == [0, true_delay_samples]
assert received_clean.size == source.size + impulse_response.size - 1
assert received_noisy.shape == received_clean.shape

print(f"True delay: {true_delay_samples} samples = {TRUE_DELAY_S * 1e3:.3f} ms")
print(f"Source samples: {source.size}; received samples: {received_noisy.size}")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 8), constrained_layout=True)
axes[0].plot(source_time_s * 1e3, instantaneous_frequency_hz / 1e3, linewidth=1.2)
axes[0].set(title="Dolphin-like nonlinear frequency contour", xlabel="Time (ms)", ylabel="Frequency (kHz)")

filter_time_ms = np.arange(impulse_response.size) / FS * 1e3
axes[1].stem(filter_time_ms, impulse_response, basefmt=" ")
axes[1].set(title="Two-path impulse response", xlabel="Delay (ms)", ylabel="Gain")

axes[2].plot(received_time_s * 1e3, received_clean, label="Clean", linewidth=1.0)
axes[2].plot(received_time_s * 1e3, received_noisy, label="Noisy", linewidth=0.7, alpha=0.65)
axes[2].set(title="Received signal", xlabel="Time (ms)", ylabel="Amplitude")
axes[2].legend()
plt.show()

## Extract the contour and reconstruct the analytic template

The native Python extractor receives the generated WAV and the manually supplied suspected interval. It returns a signed enhanced-spectrogram ridge in memory and also writes CSV, NPZ, JSON, and diagnostic PNG artifacts. The ridge is fitted to the known synthetic $\tan(x)-\sin(x)+1$ family with clipped path scores as confidence weights, then integrated into phase on the 48 kHz audio grid.


In [ ]:
contour_cfg = birdcall_contour_default_config()
contour_cfg.output.dir = CONTOUR_OUTPUT_DIR
contour_cfg.output.prefix = "synthetic_dolphin"
contour_cfg.debug.verbose = True

contour_results = extract_birdcall_contours(
    wav_path,
    [(0.0, SIGNAL_DURATION_S)],
    contour_cfg,
)
if not contour_results.contours:
    statuses = ", ".join(
        f"interval {item.interval_index}: {item.status} {item.error_message}".strip()
        for item in contour_results.interval_statuses
    )
    raise RuntimeError(f"No contour was accepted. Extraction status: {statuses}")

extracted_contour = contour_results.contours[0]
contour_data = {
    "time_seconds": extracted_contour.time_sec_abs,
    "frequency_hz": extracted_contour.freq_hz,
    "path_scores": extracted_contour.path_scores,
    "requested_start_seconds": extracted_contour.requested_start_sec,
    "requested_end_seconds": extracted_contour.requested_end_sec,
    "sample_rate_hz": contour_results.fs,
}


def validate_contour_data(contour):
    """Validate the accepted extractor contour used by this experiment."""
    array_lengths = {
        contour["time_seconds"].size,
        contour["frequency_hz"].size,
        contour["path_scores"].size,
    }
    if array_lengths == {0} or len(array_lengths) != 1:
        raise ValueError("Contour time, frequency, and score arrays must be nonempty and equal length.")
    for name in ("time_seconds", "frequency_hz", "path_scores"):
        if not np.all(np.isfinite(contour[name])):
            raise ValueError(f"Contour field {name} contains NaN or infinite values.")
    if not np.all(np.diff(contour["time_seconds"]) > 0):
        raise ValueError("Contour timestamps must be strictly increasing.")
    if not np.all(contour["frequency_hz"] > 0):
        raise ValueError("Contour frequencies must be positive.")
    # path_scores are signed robust-enhancement values, not probabilities.
    if contour["requested_end_seconds"] <= contour["requested_start_seconds"]:
        raise ValueError("The requested contour interval must have positive duration.")
    return contour


def reconstruct_analytic_template(
    contour, expected_fs, x_min, x_max, taper_alpha
):
    """Fit the synthetic contour family and return a unit-energy analytic chirp."""
    contour_fs = contour["sample_rate_hz"]
    if contour_fs != expected_fs:
        raise ValueError(
            f"Contour sampling rate ({contour_fs} Hz) does not match FS ({expected_fs} Hz)."
        )

    start_s = contour["requested_start_seconds"]
    end_s = contour["requested_end_seconds"]
    sample_count = round((end_s - start_s) * expected_fs)
    if sample_count < 2:
        raise ValueError("The requested contour interval is too short for reconstruction.")

    template_time_s = start_s + np.arange(sample_count) / expected_fs
    template_end_s = template_time_s[-1]

    def contour_basis(time_s):
        x = x_min + (x_max - x_min) * (time_s - start_s) / (template_end_s - start_s)
        return np.tan(x) - np.sin(x) + 1

    detected_time_s = contour["time_seconds"]
    detected_frequency_hz = contour["frequency_hz"]
    design = np.column_stack((
        np.ones(detected_time_s.size), contour_basis(detected_time_s)
    ))
    weights = np.sqrt(np.maximum(contour["path_scores"], 0.1))
    coefficients, *_ = np.linalg.lstsq(
        design * weights[:, None], detected_frequency_hz * weights, rcond=None
    )
    fitted_detected_frequency_hz = design @ coefficients
    fitted_frequency_hz = coefficients[0] + coefficients[1] * contour_basis(template_time_s)

    if not np.all(np.isfinite(fitted_frequency_hz)):
        raise ValueError("Fitted contour contains NaN or infinite frequencies.")
    if np.min(fitted_frequency_hz) <= 0:
        raise ValueError("Fitted contour contains non-positive frequencies.")
    if np.max(fitted_frequency_hz) >= expected_fs / 2:
        raise ValueError("Fitted contour reaches or exceeds the Nyquist frequency.")

    phase_rad = np.zeros(sample_count)
    phase_rad[1:] = 2 * np.pi * np.cumsum(
        0.5 * (fitted_frequency_hz[:-1] + fitted_frequency_hz[1:]) / expected_fs
    )
    envelope = signal.windows.tukey(sample_count, alpha=taper_alpha)
    template = envelope * np.exp(1j * phase_rad)
    template /= np.linalg.norm(template)

    return {
        "template": template,
        "template_time_seconds": template_time_s,
        "fitted_frequency_hz": fitted_frequency_hz,
        "fitted_detected_frequency_hz": fitted_detected_frequency_hz,
        "fit_coefficients": coefficients,
    }


contour_data = validate_contour_data(contour_data)
template_reconstruction = reconstruct_analytic_template(
    contour_data, FS, SIGNAL_X_MIN, SIGNAL_X_MAX, WHISTLE_TAPER_ALPHA
)
s_tilde = template_reconstruction["template"]
s_tilde_fs = contour_data["sample_rate_hz"]

print(
    f"Extracted {contour_data['time_seconds'].size} contour points; "
    f"reconstructed {s_tilde.size} complex samples at {s_tilde_fs} Hz."
)
print(
    f"Fitted frequency range: "
    f"{template_reconstruction['fitted_frequency_hz'].min():.1f}--"
    f"{template_reconstruction['fitted_frequency_hz'].max():.1f} Hz"
)
print(f"Contour artifacts: {contour_results.output_dir}")


## Contour fit and FFT matched-filter estimator

The fitted curve suppresses frame-level ridge error before frequency is integrated into phase. Signed enhancement scores are clipped only when converted to fit weights. The estimator forms the conjugate-reversed matched-filter kernel explicitly and applies it only to `received_noisy` with `fftconvolve`. A one-time direct-convolution comparison verifies numerical equivalence.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5), constrained_layout=True)
ax.scatter(
    contour_data["time_seconds"] * 1e3,
    contour_data["frequency_hz"] / 1e3,
    s=12, alpha=0.65, label="Python extracted ridge",
)
ax.plot(
    contour_data["time_seconds"] * 1e3,
    template_reconstruction["fitted_detected_frequency_hz"] / 1e3,
    linewidth=1.5, label="Confidence-weighted fit",
)
ax.plot(
    template_reconstruction["template_time_seconds"] * 1e3,
    template_reconstruction["fitted_frequency_hz"] / 1e3,
    linestyle="--", linewidth=1.0, label="Reconstructed audio-rate contour",
)
ax.set(
    title="Extracted contour and fitted analytic-template trajectory",
    xlabel="Time (ms)", ylabel="Frequency (kHz)",
)
ax.legend()
plt.show()


def validate_template(template, template_fs, expected_fs):
    """Return a validated real or complex template or raise a descriptive error."""
    if template is None:
        raise ValueError(
            "s_tilde is missing. Reconstruct the analytic template before estimating delay."
        )
    if template_fs is None:
        raise ValueError("s_tilde_fs is missing; specify the template sampling rate.")
    if template_fs != expected_fs:
        raise ValueError(
            f"Template sampling rate ({template_fs} Hz) does not match FS ({expected_fs} Hz)."
        )

    template = np.asarray(template)
    if template.ndim != 1:
        raise ValueError(f"s_tilde must be one-dimensional; received shape {template.shape}.")
    if template.size == 0:
        raise ValueError("s_tilde must contain at least one sample.")
    if not np.issubdtype(template.dtype, np.number):
        raise ValueError("s_tilde must contain numeric samples.")

    target_dtype = np.complex128 if np.iscomplexobj(template) else float
    template = template.astype(target_dtype, copy=False)
    if not np.all(np.isfinite(template)):
        raise ValueError("s_tilde contains NaN or infinite samples.")
    if np.linalg.norm(template) == 0:
        raise ValueError("s_tilde must not be identically zero.")
    return template


def estimate_slant_delay(
    received, template, fs, min_peak_separation_s, max_delay_s, min_prominence_ratio
):
    """Estimate echo delay using a conjugate-reversed FFT matched filter."""
    received = np.asarray(received, dtype=float)
    if received.ndim != 1 or received.size == 0 or not np.all(np.isfinite(received)):
        raise ValueError("received must be a nonempty, finite, one-dimensional array.")
    if fs <= 0:
        raise ValueError("fs must be positive.")
    if not 0 < min_peak_separation_s < max_delay_s:
        raise ValueError("Require 0 < min_peak_separation_s < max_delay_s.")
    if not 0 < min_prominence_ratio < 1:
        raise ValueError("min_prominence_ratio must lie strictly between 0 and 1.")

    template_energy = float(np.sum(np.abs(template) ** 2))
    matched_kernel = np.conj(template[::-1]) / template_energy
    response = signal.fftconvolve(received, matched_kernel, mode="full")
    lags_samples = np.arange(-(template.size - 1), received.size, dtype=int)

    direct_response = signal.convolve(
        received, matched_kernel, mode="full", method="direct"
    )
    convolution_check_max_abs_error = float(np.max(np.abs(response - direct_response)))
    if not np.allclose(response, direct_response, rtol=1e-10, atol=1e-10):
        raise RuntimeError("FFT and direct matched-filter convolution results disagree.")

    response_magnitude = np.abs(response)
    direct_peak_index = int(np.argmax(response_magnitude))
    direct_lag = int(lags_samples[direct_peak_index])

    minimum_distance_samples = max(1, round(min_peak_separation_s * fs))
    maximum_delay_samples = round(max_delay_s * fs)
    prominence = min_prominence_ratio * response_magnitude[direct_peak_index]
    peaks, _ = signal.find_peaks(
        response_magnitude, distance=minimum_distance_samples, prominence=prominence
    )
    candidate_mask = (
        (lags_samples[peaks] >= direct_lag + minimum_distance_samples)
        & (lags_samples[peaks] <= direct_lag + maximum_delay_samples)
    )
    echo_candidates = peaks[candidate_mask]
    if echo_candidates.size == 0:
        raise RuntimeError(
            "No credible delayed peak was found after the direct peak. Check s_tilde "
            "or adjust the separation, maximum-delay, and prominence settings."
        )

    echo_peak_index = int(echo_candidates[np.argmax(response_magnitude[echo_candidates])])
    delay_samples = int(lags_samples[echo_peak_index] - direct_lag)
    return {
        "response": response,
        "matched_kernel": matched_kernel,
        "lags_samples": lags_samples,
        "candidate_peak_indices": echo_candidates,
        "direct_peak_index": direct_peak_index,
        "echo_peak_index": echo_peak_index,
        "delay_samples": delay_samples,
        "delay_seconds": delay_samples / fs,
        "convolution_check_max_abs_error": convolution_check_max_abs_error,
    }

In [ ]:
validated_s_tilde = validate_template(s_tilde, s_tilde_fs, FS)
estimate = estimate_slant_delay(
    received_noisy,
    validated_s_tilde,
    FS,
    MIN_PEAK_SEPARATION_S,
    MAX_DELAY_S,
    MIN_PROMINENCE_RATIO,
)

estimated_delay_samples = estimate["delay_samples"]
estimated_delay_s = estimate["delay_seconds"]
absolute_error_samples = abs(estimated_delay_samples - true_delay_samples)
absolute_error_s = abs(estimated_delay_s - TRUE_DELAY_S)

assert estimated_delay_samples == (
    estimate["lags_samples"][estimate["echo_peak_index"]]
    - estimate["lags_samples"][estimate["direct_peak_index"]]
)
assert absolute_error_samples <= MAX_DELAY_ERROR_SAMPLES, (
    f"Delay error {absolute_error_samples} samples exceeds the "
    f"{MAX_DELAY_ERROR_SAMPLES}-sample acceptance tolerance."
)

print(f"True delay:      {true_delay_samples:4d} samples = {TRUE_DELAY_S:.6f} s = {TRUE_DELAY_S * 1e3:.3f} ms")
print(f"Estimated delay: {estimated_delay_samples:4d} samples = {estimated_delay_s:.6f} s = {estimated_delay_s * 1e3:.3f} ms")
print(f"Absolute error:  {absolute_error_samples:4d} samples = {absolute_error_s:.6f} s = {absolute_error_s * 1e3:.3f} ms")
print(
    f"FFT/direct convolution max difference: "
    f"{estimate['convolution_check_max_abs_error']:.3e}"
)

In [ ]:
lags_relative_to_direct_ms = (
    estimate["lags_samples"]
    - estimate["lags_samples"][estimate["direct_peak_index"]]
) / FS * 1e3
response_magnitude = np.abs(estimate["response"])
response_magnitude /= response_magnitude.max()

fig, ax = plt.subplots(figsize=(11, 4.5), constrained_layout=True)
ax.plot(lags_relative_to_direct_ms, response_magnitude, linewidth=1.0)
for label, index, color in (
    ("Direct", estimate["direct_peak_index"], "tab:green"),
    ("Delayed", estimate["echo_peak_index"], "tab:red"),
):
    x_ms = lags_relative_to_direct_ms[index]
    y = response_magnitude[index]
    ax.scatter(x_ms, y, color=color, zorder=3, label=f"{label}: {x_ms:.3f} ms")
    ax.axvline(x_ms, color=color, linestyle="--", alpha=0.65)
ax.set_xlim(-5, MAX_DELAY_S * 1e3 + 5)
ax.set(
    title="FFT matched-filter response and selected arrivals",
    xlabel="Lag relative to direct arrival (ms)",
    ylabel="Normalized magnitude",
)
ax.legend()
plt.show()